## Feature Extraction

- 사전 학습된 모델의 가중치를 고정하고, 새로운 작업에 맞게 마지막 레이어만 재학습하는 기법 
- 모델의 초기 레이어에서 추출한 저수준의 특징(가장자리, 텍스처 등)을 그대로 활용
    - 이미지 분류 등 다양한 작업에서 공통적으로 유용, 새로운 데이터셋에서도 효과적으로 활용 될 수 있다. 
    - 복잡한 레이어들을 고정한 채, 마지막 완전 연결층(FC Layer)만 재학습, 효율적으로 새로운 작업을 해결 

### 1. 특징 추출기 

- 일반적으로 모델의 여러 컨볼루션 레이어(합성곱)와 풀링레이어로 구성 
- 이미지에서 색상, 텍스처, 모양 등과 같은 저수준(low-level) 특징들을 인식하는데 사용 


### 1.1 특징 추출기(feturea extractor) 접근 방법 적용

#### 특징 추출기(feature extraction) 방법을 적용할 때, 단계

1. pooling 네모칸 가중치 조정: 학습 과정에서 업데이트 되지 않도록 설정 
2. 완전 연결층(컨볼루션 + 렐루)를 교체하거나 수정, 예를 들어, 새로운 분류작업에 맞춰 출력 레이어의 뉴런수를 조정, 새로운 활성화 함수를 적용 
3. 학습 파라미터 조정: 고정되지 않은 레이어(완전 연결층)에 대해서만 가중치가 업데이트되도록 학습 파라미터를 설정 

- 이러한 접근 방식은 새로운 데이터셋이 기존에 학습된 데이터셋과 유사할 때 효과적
- 기본적인 이미지 특성이 크게 변경되지 않은 경우
- 이미 잘 학습된 특징 추출기를 그대로 활용하여 빠르고 효율적으로 새로운 작업을 학습할 수 있다. 
- 학습시간을 단축, 필요한 계산자원을 절약하는 동시에 과적합 위험을 감소 시킨다. 

##### 장점

- 데이터가 적을 때 유리 
- 학습 시간 단축
- 모델의 일반화 능력 

##### 한계

- 최적의 성능을 보장하지 않음: 고정된 가중치로 인해 최적의 성능을 얻지 못할 수 있다.
- 새로운 데이터셋과의 차이: 사전에 학습된 데이터셋과 새로운 데이터셋이 매우 다른경우, 특징 추출의 효과가 줄어 들 수 있다. 

In [1]:
import torch
import torchvision.models as models

model = models.resnet18(pretrained=True)
print(model)

/Users/donghun2/workspace/machine_learning/dacon/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/donghun2/workspace/machine_learning/dacon/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [2]:
for name, param in model.named_parameters():
    print(f'{name}: {param.requires_grad}')

conv1.weight: True
bn1.weight: True
bn1.bias: True
layer1.0.conv1.weight: True
layer1.0.bn1.weight: True
layer1.0.bn1.bias: True
layer1.0.conv2.weight: True
layer1.0.bn2.weight: True
layer1.0.bn2.bias: True
layer1.1.conv1.weight: True
layer1.1.bn1.weight: True
layer1.1.bn1.bias: True
layer1.1.conv2.weight: True
layer1.1.bn2.weight: True
layer1.1.bn2.bias: True
layer2.0.conv1.weight: True
layer2.0.bn1.weight: True
layer2.0.bn1.bias: True
layer2.0.conv2.weight: True
layer2.0.bn2.weight: True
layer2.0.bn2.bias: True
layer2.0.downsample.0.weight: True
layer2.0.downsample.1.weight: True
layer2.0.downsample.1.bias: True
layer2.1.conv1.weight: True
layer2.1.bn1.weight: True
layer2.1.bn1.bias: True
layer2.1.conv2.weight: True
layer2.1.bn2.weight: True
layer2.1.bn2.bias: True
layer3.0.conv1.weight: True
layer3.0.bn1.weight: True
layer3.0.bn1.bias: True
layer3.0.conv2.weight: True
layer3.0.bn2.weight: True
layer3.0.bn2.bias: True
layer3.0.downsample.0.weight: True
layer3.0.downsample.1.weight: T

In [3]:
for name, param in model.named_parameters():
    print(f'{name}: {param.requires_grad}')

conv1.weight: True
bn1.weight: True
bn1.bias: True
layer1.0.conv1.weight: True
layer1.0.bn1.weight: True
layer1.0.bn1.bias: True
layer1.0.conv2.weight: True
layer1.0.bn2.weight: True
layer1.0.bn2.bias: True
layer1.1.conv1.weight: True
layer1.1.bn1.weight: True
layer1.1.bn1.bias: True
layer1.1.conv2.weight: True
layer1.1.bn2.weight: True
layer1.1.bn2.bias: True
layer2.0.conv1.weight: True
layer2.0.bn1.weight: True
layer2.0.bn1.bias: True
layer2.0.conv2.weight: True
layer2.0.bn2.weight: True
layer2.0.bn2.bias: True
layer2.0.downsample.0.weight: True
layer2.0.downsample.1.weight: True
layer2.0.downsample.1.bias: True
layer2.1.conv1.weight: True
layer2.1.bn1.weight: True
layer2.1.bn1.bias: True
layer2.1.conv2.weight: True
layer2.1.bn2.weight: True
layer2.1.bn2.bias: True
layer3.0.conv1.weight: True
layer3.0.bn1.weight: True
layer3.0.bn1.bias: True
layer3.0.conv2.weight: True
layer3.0.bn2.weight: True
layer3.0.bn2.bias: True
layer3.0.downsample.0.weight: True
layer3.0.downsample.1.weight: T

## 모든 레이어의 가중치를 고정하여 학습 하지 않도록 설정


In [4]:
for param in model.parameters():
    param.requires_grad = False

In [5]:
for name, param in model.named_parameters():
    print(f'{name}: {param.requires_grad}')

conv1.weight: False
bn1.weight: False
bn1.bias: False
layer1.0.conv1.weight: False
layer1.0.bn1.weight: False
layer1.0.bn1.bias: False
layer1.0.conv2.weight: False
layer1.0.bn2.weight: False
layer1.0.bn2.bias: False
layer1.1.conv1.weight: False
layer1.1.bn1.weight: False
layer1.1.bn1.bias: False
layer1.1.conv2.weight: False
layer1.1.bn2.weight: False
layer1.1.bn2.bias: False
layer2.0.conv1.weight: False
layer2.0.bn1.weight: False
layer2.0.bn1.bias: False
layer2.0.conv2.weight: False
layer2.0.bn2.weight: False
layer2.0.bn2.bias: False
layer2.0.downsample.0.weight: False
layer2.0.downsample.1.weight: False
layer2.0.downsample.1.bias: False
layer2.1.conv1.weight: False
layer2.1.bn1.weight: False
layer2.1.bn1.bias: False
layer2.1.conv2.weight: False
layer2.1.bn2.weight: False
layer2.1.bn2.bias: False
layer3.0.conv1.weight: False
layer3.0.bn1.weight: False
layer3.0.bn1.bias: False
layer3.0.conv2.weight: False
layer3.0.bn2.weight: False
layer3.0.bn2.bias: False
layer3.0.downsample.0.weight: 

In [6]:
input_size = model.fc.in_features
output_size = model.fc.out_features

print(f'fc 레이어의 입력 크기: ',input_size)
print(f'fc 레이어의 출력 크기: ',output_size)

fc 레이어의 입력 크기:  512
fc 레이어의 출력 크기:  1000


In [7]:
num_features = model.fc.in_features  
model.fc = torch.nn.Linear(num_features, 10) 
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [8]:
for name, param in model.named_parameters():
    print(f'{name}: {param.requires_grad}')

conv1.weight: False
bn1.weight: False
bn1.bias: False
layer1.0.conv1.weight: False
layer1.0.bn1.weight: False
layer1.0.bn1.bias: False
layer1.0.conv2.weight: False
layer1.0.bn2.weight: False
layer1.0.bn2.bias: False
layer1.1.conv1.weight: False
layer1.1.bn1.weight: False
layer1.1.bn1.bias: False
layer1.1.conv2.weight: False
layer1.1.bn2.weight: False
layer1.1.bn2.bias: False
layer2.0.conv1.weight: False
layer2.0.bn1.weight: False
layer2.0.bn1.bias: False
layer2.0.conv2.weight: False
layer2.0.bn2.weight: False
layer2.0.bn2.bias: False
layer2.0.downsample.0.weight: False
layer2.0.downsample.1.weight: False
layer2.0.downsample.1.bias: False
layer2.1.conv1.weight: False
layer2.1.bn1.weight: False
layer2.1.bn1.bias: False
layer2.1.conv2.weight: False
layer2.1.bn2.weight: False
layer2.1.bn2.bias: False
layer3.0.conv1.weight: False
layer3.0.bn1.weight: False
layer3.0.bn1.bias: False
layer3.0.conv2.weight: False
layer3.0.bn2.weight: False
layer3.0.bn2.bias: False
layer3.0.downsample.0.weight: 

```python
import torch
import torch.optim as optim
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import torchvision.models as models
import numpy as np
from torchvision.datasets import ImageFolder

# CIFAR-10 데이터셋 전처리
transform = transforms.Compose([
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# CIFAR-10 전체 데이터셋 로드
full_dataset = ImageFolder(root='./train', transform=transform)

# 데이터셋에서 순서대로 1000개 샘플 선택
subset_indices = np.arange(1000)  
train_subset = Subset(full_dataset, subset_indices)
train_loader = DataLoader(train_subset, batch_size=8, shuffle=True)
```

```python
model = models.resnet18(pretrained=True)
for param in model.parameters():
    param.requires_grad = False  

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)

input_size = model.fc.in_features
output_size = model.fc.out_features

print(f'fc 레이어의 입력 크기: ',input_size)
print(f'변경된 fc 레이어의 출력 크기: ',output_size)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)  

for epoch in range(3):  
    model.train()
    running_loss = 0.0
    correct = 0  
    total = 0  

    for inputs, labels in train_loader:
        optimizer.zero_grad()  

        # 전방향 전달
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # 역방향 전달 및 최적화
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # 정확도 계산
        _, predicted = torch.max(outputs, 1)  
        total += labels.size(0) 
        correct += (predicted == labels).sum().item()

    # 에포크당 손실 및 정확도 출력
    epoch_loss = running_loss / len(train_loader)
    accuracy = 100 * correct / total
    print(f'Epoch {epoch+1}, Loss: {epoch_loss:.4f}, Accuracy: {accuracy:.2f}%')

print('학습 완료')
```